# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Final Pyoyect Part 1: Batch Processing** </center>
---
**Profesor**: Pablo Camarillo Ramirez
---
**Student**: Nicolas Navarro Valenzuela 746812

In [19]:
from spark_utils import SparkUtils

#"spark-submit --master yarn --deploy-mode cluster --packages org.apache.hadoop:hadoop-aws:3.3.4 /home/hadoop/mi_pipeline.py"

In [20]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# ── Sesión Spark ────────────────────────────────────────────────────────────
spark = SparkSession.builder \
    .appName("BigData-Final-Pipeline-Local") \
    .getOrCreate()


In [21]:
!ls -lah /opt/spark/work-dir/data/batch_data/part-0000.json

-rwxrwxrwx 1 root root 402M Apr 25 18:41 /opt/spark/work-dir/data/batch_data/part-0000.json


In [22]:
DATA_PATH = "/opt/spark/work-dir/data/batch_data/part-0000.json"
OUTPUT_PATH = "/opt/spark/work-dir/data/output/"

#  1. Lectura de datos
df_raw = spark.read.json(DATA_PATH)
print(f"Datos cargados desde: {DATA_PATH}")
print(f"Total de registros: {df_raw.count():,}")
df_raw.show(5)

Datos cargados desde: /opt/spark/work-dir/data/batch_data/part-0000.json


Total de registros: 500,000
+-----------+----------------+--------------------+--------------------+----------------+------------+-----------+----------+-------------------+--------------+--------------------+--------------------+--------+------------+--------------------+----------------+----------------+---------+------------+--------------------+----------+------------+
|   category|customer_country|      customer_email|         customer_id|   customer_name|discount_pct|is_returned|order_date|    order_timestamp|payment_method|          product_id|        product_name|quantity|review_score|         review_text|   shipping_city|shipping_country|   status|total_amount|      transaction_id|unit_price|warehouse_id|
+-----------+----------------+--------------------+--------------------+----------------+------------+-----------+----------+-------------------+--------------+--------------------+--------------------+--------+------------+--------------------+----------------+--------------

In [23]:
#  2. LIMPIEZA (Transformación 1) 
df_clean = df_raw.dropDuplicates(["transaction_id"]) \
                 .dropna(subset=["transaction_id", "customer_id", 
                                 "category", "total_amount"])
print(f"Registros después de limpieza: {df_clean.count():,}")
print("Duplicados eliminados:", df_raw.count() - df_clean.count())

Registros después de limpieza: 500,000


Duplicados eliminados: 0


In [24]:
#  3. COLUMNA DERIVADA (Transformación 2) 
df_enriched = df_clean.withColumn(
    "revenue_after_discount",
    F.round(
        F.col("total_amount") * (1 - F.col("discount_pct") / 100), 2
    )
).withColumn(
    "order_year",
    F.year(F.to_date(F.col("order_date")))
)
print("Columnas derivadas agregadas: revenue_after_discount, order_year")
df_enriched.select("total_amount", "discount_pct", "revenue_after_discount", "order_year").show(5)

Columnas derivadas agregadas: revenue_after_discount, order_year


+------------+------------+----------------------+----------+
|total_amount|discount_pct|revenue_after_discount|order_year|
+------------+------------+----------------------+----------+
|    20598.71|       11.88|              18151.58|      2025|
|    25247.57|        3.91|              24260.39|      2025|
|     7353.93|       39.77|               4429.27|      2025|
|    20801.81|       17.64|              17132.37|      2024|
|    37616.69|       26.26|              27738.55|      2026|
+------------+------------+----------------------+----------+
only showing top 5 rows


In [25]:
#  4. FILTRADO (Transformación 3) 
df_filtered = df_enriched.filter(
    (F.col("status") != "cancelled") & (F.col("total_amount") > 0)
)
print(f"Registros después de filtrado: {df_filtered.count():,}")
print(f"Registros filtrados: {df_enriched.count() - df_filtered.count():,}")

Registros después de filtrado: 374,918


Registros filtrados: 125,082


In [26]:
#  5. AGREGACIÓN (Transformación 4)
df_agg = df_filtered.groupBy("category", "order_year") \
    .agg(
        F.count("transaction_id").alias("total_transactions"),
        F.round(F.sum("revenue_after_discount"), 2).alias("total_revenue"),
        F.round(F.avg("review_score"), 2).alias("avg_review_score")
    )
print("Agregación por categoría y año completada")
df_agg.show(10)

Agregación por categoría y año completada


+-----------+----------+------------------+--------------+----------------+
|   category|order_year|total_transactions| total_revenue|avg_review_score|
+-----------+----------+------------------+--------------+----------------+
|   clothing|      2024|             18543|2.7820962169E8|             3.0|
|       home|      2026|              8278| 1.238340792E8|            2.99|
|       toys|      2025|             27065|4.0631512027E8|            2.99|
|     beauty|      2026|              8278|1.2456490037E8|            3.02|
|       food|      2025|             26540|4.0078279109E8|             3.0|
|       food|      2024|             18327|2.7436748408E8|             3.0|
|   clothing|      2025|             26769|4.0070459401E8|             3.0|
|       home|      2024|             18494|2.7849485448E8|            3.02|
|electronics|      2024|             18380| 2.741708892E8|             3.0|
|     beauty|      2025|             26821|4.0122487127E8|            2.99|
+-----------

In [27]:
#  6. JOIN (Transformación 5) 
df_country_stats = df_filtered.groupBy("customer_country") \
    .agg(F.count("*").alias("orders_per_country"))

df_final = df_filtered.join(df_country_stats, on="customer_country", how="left")
print("Join con estadísticas por país completado")
print(f"Registros finales: {df_final.count():,}")
df_final.select("customer_country", "orders_per_country").distinct().show(10)

Join con estadísticas por país completado


Registros finales: 374,918


+----------------+------------------+
|customer_country|orders_per_country|
+----------------+------------------+
|              MM|              1974|
|              LT|              1913|
|              DZ|              1919|
|              CI|              1917|
|              AZ|              1958|
|              SC|              1951|
|              FI|              1910|
|              UA|              1931|
|              KI|              1866|
|              ZM|              1914|
+----------------+------------------+
only showing top 10 rows


In [28]:
#  7. PERSISTENCIA 
df_final.write \
    .mode("overwrite") \
    .partitionBy("category") \
    .parquet(OUTPUT_PATH + "transactions_processed/")

print(f"Pipeline completado exitosamente!")
print(f"Datos guardados en: {OUTPUT_PATH}transactions_processed/")

26/04/26 20:23:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/04/26 20:23:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/04/26 20:23:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/04/26 20:23:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/04/26 20:23:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/04/26 20:23:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/04/26 20:23:01 WARN MemoryManager: Total allocation exceeds 95.

Pipeline completado exitosamente!
Datos guardados en: /opt/spark/work-dir/data/output/transactions_processed/


In [29]:
#  VISUALIZACIÓN DE RESULTADOS 
print(f"\n Schema de los datos procesados:")
df_final.printSchema()
print(f"\n Muestra de 10 registros finales:")
df_final.select("transaction_id", "customer_name", "category", "order_year", 
                "total_amount", "revenue_after_discount", "orders_per_country").show(10, truncate=False)


 Schema de los datos procesados:
root
 |-- customer_country: string (nullable = true)
 |-- category: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: long (nullable = true)
 |-- review_score: long (nullable = true)
 |-- review_text: string (nullable = true)
 |-- shipping_city: string (nullable = true)
 |-- shipping_country: string (nullable = true)
 |-- status: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- warehouse_id: long (nullable = true)
 |

+------------------------------------+------------------+-----------+----------+------------+----------------------+------------------+
|transaction_id                      |customer_name     |category   |order_year|total_amount|revenue_after_discount|orders_per_country|
+------------------------------------+------------------+-----------+----------+------------+----------------------+------------------+
|0003660f-519d-4151-bbc5-117f8929dc12|Joseph Thornton   |electronics|2025      |20598.71    |18151.58              |1988              |
|00061a06-6fc2-45ef-be4c-c42f8b4f696a|Michael Mayer     |sports     |2025      |7353.93     |4429.27               |1862              |
|000e4471-d5f0-4236-b6d3-887ed65cf223|Travis Morris     |sports     |2025      |1499.98     |952.79                |1906              |
|0012d2b9-8e49-4ab4-b203-27882c6210a0|Virginia Anderson |food       |2025      |21109.6     |20425.65              |1939              |
|0018f1ad-54c3-4a14-918f-b58d66573b28|Lisa Morro

In [ ]:
spark.stop()